# Notebook overview

This notebook is the working transcript pipeline for the Manfluencer Project.

The design is now explicitly two-pass:

1. local ASR and local diarization create the rough transcript
2. optional Gemini grounding and chunk refinement repair speaker boundaries, proper names, and documentary/interview structure

The final saved artifact is still one plain-text transcript file per run.


# Install / environment notes

This notebook is optimized for Apple Silicon and uses a staged pipeline:

1. local ASR with word timestamps
2. local pyannote diarization
3. optional Gemini grounding from the public YouTube URL
4. optional Gemini chunk refinement for speaker repair and transcript cleanup

Required local setup:

- `ffmpeg` and `yt-dlp` on your PATH
- Python `3.11` Jupyter kernel
- `HF_TOKEN` in `.env` for `pyannote/speaker-diarization-community-1`
- optional `GEMINI_API_KEY` or `GOOGLE_API_KEY` in `.env` for the Gemini refinement stage

Recommended install:

```bash
brew install ffmpeg yt-dlp
python -m pip install -r requirements.txt
```

If the Gemini key is missing, the notebook still runs end-to-end locally and skips the Gemini context/refinement stages.


# Imports and configuration


In [ ]:
from __future__ import annotations

import importlib
import json
import logging
import os
import re
import shutil
import subprocess
from pathlib import Path
from textwrap import dedent
from typing import Any, Optional, TypedDict

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger("mac_youtube_transcript_pipeline")

ROOT = Path.cwd()


def load_dotenv_file(dotenv_path: Path) -> None:
    if not dotenv_path.exists():
        return

    for raw_line in dotenv_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)


load_dotenv_file(ROOT / ".env")


def require_binary(name: str) -> str:
    path = shutil.which(name)
    if not path:
        raise RuntimeError(
            f"Required binary '{name}' was not found on PATH. Install it first and re-run the notebook."
        )
    return path


def run_command(command: list[str], *, check: bool = True, capture_output: bool = True) -> subprocess.CompletedProcess:
    log.info("Running command: %s", " ".join(command))
    return subprocess.run(
        command,
        check=check,
        capture_output=capture_output,
        text=True,
    )


def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text or "").strip()
    text = re.sub(r"\s+([,.;:!?%])", r"\1", text)
    text = re.sub(r"([\(\[\{])\s+", r"\1", text)
    text = re.sub(r"\s+([\)\]\}])", r"\1", text)
    text = re.sub(r"(?<=[A-Za-z])\s+-\s+(?=[A-Za-z])", "-", text)
    return text.strip()


def stable_dedupe(items: list[str]) -> list[str]:
    seen = set()
    ordered = []
    for item in items:
        value = clean_text(str(item))
        if not value:
            continue
        key = value.casefold()
        if key in seen:
            continue
        seen.add(key)
        ordered.append(value)
    return ordered


def safe_slug(text: str, fallback: str = "audio") -> str:
    text = re.sub(r"\[[^\]]+\]", "", text)
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text).strip("._-")
    return text[:120] or fallback


def safe_transcript_filename(title: str, fallback: str = "transcript") -> str:
    title = clean_text(title or "")
    title = title.replace('"', "")
    title = re.sub(r"[\\/:*?<>|]+", "", title)
    title = re.sub(r"\s+", " ", title).strip().rstrip(".")
    return f"{title or fallback}.txt"


def format_title_line(title: str) -> str:
    normalized = clean_text(title or "Untitled")
    if '"' in normalized:
        return normalized
    return f'"{normalized}"'


def pretty_count(value: Any) -> str:
    try:
        return f"{int(value):,}"
    except (TypeError, ValueError):
        return str(value)


def infer_primary_speaker_name(metadata: dict, explicit_name: Optional[str] = None) -> Optional[str]:
    if explicit_name:
        return explicit_name

    title = (metadata.get("title") or "").strip()
    uploader = (metadata.get("uploader") or "").strip()
    channel = (metadata.get("channel") or "").strip()

    title_match = re.search(r"\s-\s([^\-]+)$", title)
    if title_match:
        candidate = clean_text(title_match.group(1))
        if candidate and len(candidate.split()) <= 6:
            return candidate

    for field in (uploader, channel):
        if not field:
            continue
        if "youtube" in field.lower():
            continue
        if len(field.split()) <= 6:
            return field

    return None


def build_stats_line(metadata: dict) -> Optional[str]:
    stats = []
    if metadata.get("view_count") is not None:
        stats.append(f"Views: {pretty_count(metadata['view_count'])}")
    if metadata.get("like_count") is not None:
        stats.append(f"Likes: {pretty_count(metadata['like_count'])}")
    if metadata.get("comment_count") is not None:
        stats.append(f"Comments: {pretty_count(metadata['comment_count'])}")
    if not stats:
        return None
    return "Stats: " + "; ".join(stats) + "."


def resolve_api_key() -> str:
    return (GEMINI_API_KEY or GOOGLE_API_KEY or "").strip()


def extract_gemini_text(response: Any) -> str:
    direct_text = getattr(response, "text", None)
    if direct_text:
        return str(direct_text)

    pieces = []
    for candidate in getattr(response, "candidates", None) or []:
        content = getattr(candidate, "content", None)
        for part in getattr(content, "parts", None) or []:
            part_text = getattr(part, "text", None)
            if part_text:
                pieces.append(str(part_text))
    return "\n".join(pieces).strip()


def detect_notebook_package(name: str) -> bool:
    try:
        importlib.import_module(name)
        return True
    except Exception:
        return False


print(f"Working directory: {ROOT}")
print("ffmpeg:", shutil.which("ffmpeg") or "missing")
print("yt_dlp:", detect_notebook_package("yt_dlp"))
print("pyannote.audio:", detect_notebook_package("pyannote.audio"))
print("mlx_whisper:", detect_notebook_package("mlx_whisper"))
print("faster_whisper:", detect_notebook_package("faster_whisper"))
print("google.genai:", detect_notebook_package("google.genai"))


# User inputs


In [ ]:
YOUTUBE_URL = ""
OUTPUT_DIR = "Generated Transcripts"
OUTPUT_SUBDIR = ""
TEMP_DIR = "temp"
AUDIO_WAV_PATH = None
WHISPER_MODEL = "large-v3-turbo"
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "").strip()
NUM_SPEAKERS = None
MIN_SPEAKERS = None
MAX_SPEAKERS = None
PRIMARY_SPEAKER_NAME = None
KNOWN_SPEAKERS = []
SPEAKER_STRATEGY = "default"
HOST_NAME = None
GUEST_NAME = None
SECONDARY_SPEAKER_NAME = None
STRICT_KNOWN_SPEAKERS = False
CONTENT_FORMAT = "auto"
TERM_HINTS = []
CONTEXT_NOTES = ""
KEEP_TEMP = False
TRANSCRIPT_FILENAME = "auto"

# Optional convenience fields for local workflows.
LOCAL_AUDIO_PATH = ""
ASR_BACKEND = "mlx_whisper"  # auto, mlx_whisper, faster_whisper
USE_GEMINI_VIDEO_CONTEXT = True
USE_GEMINI_REFINEMENT = True
GEMINI_MODEL = "gemini-2.5-flash"
GEMINI_CONTEXT_MODEL = "gemini-2.5-flash"
GEMINI_CHUNK_TARGET_CHARS = 9000
GEMINI_MAX_OUTPUT_TOKENS = 8192
GEMINI_ENABLE_SAFETY_OVERRIDE = True

output_dir = (ROOT / OUTPUT_DIR).resolve()
if OUTPUT_SUBDIR:
    output_dir = (output_dir / OUTPUT_SUBDIR).resolve()
temp_dir = (ROOT / TEMP_DIR).resolve()
output_dir.mkdir(parents=True, exist_ok=True)
temp_dir.mkdir(parents=True, exist_ok=True)

artifacts_to_cleanup: list[Path] = []
video_context: dict[str, Any] = {}

print("Output directory:", output_dir)
print("Temp directory:", temp_dir)
print("Gemini refinement enabled:", USE_GEMINI_REFINEMENT)
print("Gemini key present:", bool(GEMINI_API_KEY))


# Preset jobs


In [ ]:
VIDEO_JOBS = {
    "face_it_like_a_man": {
        "youtube_url": "https://www.youtube.com/watch?v=SoVSXTTH2dg",
        "local_audio_path": "Nigeria Audio Files/_Face it Like a Man_ - Banky Wellington [SoVSXTTH2dg].mp3",
        "primary_speaker_name": "Banky W (Olubankole Wellington)",
        "content_format": "sermon",
        "term_hints": ["Banky W", "Olubankole Wellington", "TPH", "Pastor Tony Rapu"],
        "output_subdir": "Nigeria",
    },
    "faith_after_a_fall": {
        "youtube_url": "https://www.youtube.com/watch?v=qFHXI0jHJRM",
        "local_audio_path": "Nigeria Audio Files/_Faith after a Fall_ - Banky Wellington [qFHXI0jHJRM].mp3",
        "primary_speaker_name": "Banky W (Olubankole Wellington)",
        "content_format": "sermon",
        "term_hints": ["Banky W", "Olubankole Wellington", "faith"],
        "output_subdir": "Nigeria",
    },
    "final_say_faith": {
        "youtube_url": "https://www.youtube.com/watch?v=LFs0k-eluu4",
        "local_audio_path": "Nigeria Audio Files/_Final Say Faith_ - Banky & Adesua Wellington [LFs0k-eluu4].mp3",
        "primary_speaker_name": "Banky W (Olubankole Wellington)",
        "known_speakers": [
            "Banky W (Olubankole Wellington)",
            {"name": "Adesua Wellington", "aliases": ["Adesua", "Adesua Etomi", "Adesua Etomi Wellington", "Adesua Wellington"]},
        ],
        "speaker_strategy": "primary_plus_secondary",
        "secondary_speaker_name": "Adesua Wellington",
        "content_format": "conversation_testimony",
        "term_hints": ["Banky W", "Adesua Wellington", "faith"],
        "num_speakers": 2,
        "strict_known_speakers": True,
        "output_subdir": "Nigeria",
    },
    "my_story_journey_through_hope_and_faith": {
        "youtube_url": "https://www.youtube.com/watch?v=a5PryKc1Ev8",
        "local_audio_path": "Nigeria Audio Files/_My Story - a journey through Hope & Faith_ - Banky Wellington [a5PryKc1Ev8].mp3",
        "primary_speaker_name": "Banky W (Olubankole Wellington)",
        "content_format": "sermon",
        "term_hints": ["Banky W", "Olubankole Wellington", "hope", "faith"],
        "output_subdir": "Nigeria",
    },
    "the_prison_of_pornography": {
        "youtube_url": "https://www.youtube.com/watch?v=9e9zAjM9wuA",
        "local_audio_path": "Nigeria Audio Files/_The Prison of Pornography_ - Road to Freedom Finale [9e9zAjM9wuA].mp3",
        "primary_speaker_name": "Banky W (Olubankole Wellington)",
        "content_format": "sermon",
        "term_hints": ["Banky W", "pornography", "sexual addiction"],
        "output_subdir": "Nigeria",
    },
    "andrew_kibe_071_28_commandments_of_journeying_into_wealth_health_and_respect": {
        "youtube_url": "https://www.youtube.com/watch?v=K_VkWFG1WEo",
        "local_audio_path": "Kenya Audio Files/Andrew Kibe/071 Andrew Kibe and the 28 Commandments of Journeying into Wealth, Health and Respect [K_VkWFG1WEo].mp3",
        "primary_speaker_name": "Andrew Kibe",
        "known_speakers": [
            "Andrew Kibe",
            {"name": "Joab Ogato", "aliases": ["Joab", "Joab Ogato"]},
        ],
        "speaker_strategy": "interview_guest_host",
        "host_name": "Joab Ogato",
        "guest_name": "Andrew Kibe",
        "content_format": "interview",
        "term_hints": ["Andrew Kibe", "Joab Ogato"],
        "num_speakers": 2,
        "strict_known_speakers": True,
        "output_subdir": "Kenya/Andrew Kibe",
    },
    "onyango_narelate_mens_mental_health_workshop_nakuru_january_2023": {
        "youtube_url": "https://www.youtube.com/watch?v=2fT7AehWVcs",
        "local_audio_path": "Kenya Audio Files/Onyango Otieno (Rixpoet)/#NaRelate Men's Mental Health Workshop #Nakuru #MentalHealth by Onyango Otieno  January 2023.mp3",
        "primary_speaker_name": "Onyango Otieno (Rixpoet)",
        "speaker_strategy": "primary_plus_audience",
        "content_format": "workshop",
        "term_hints": ["Onyango Otieno", "Rixpoet", "Nakuru", "mental health"],
        "min_speakers": 2,
        "max_speakers": 8,
        "output_subdir": "Kenya/Onyango Otieno (Rixpoet)",
    },
    "onyango_men_addiction_and_violence_the_story_of_our_childhood_trauma": {
        "youtube_url": "https://www.youtube.com/watch?v=5ozSU-rvJZw",
        "local_audio_path": "Kenya Audio Files/Onyango Otieno (Rixpoet)/Men, Addiction, and Violence; The Story of our Childhood Trauma. [5ozSU-rvJZw].mp3",
        "primary_speaker_name": "Onyango Otieno (Rixpoet)",
        "known_speakers": [
            "Onyango Otieno (Rixpoet)",
            {"name": "Jagero", "aliases": ["Jagero", "Uduro Jagero"]},
        ],
        "speaker_strategy": "interview_guest_host",
        "host_name": "Jagero",
        "guest_name": "Onyango Otieno (Rixpoet)",
        "content_format": "interview",
        "term_hints": ["Onyango Otieno", "Rixpoet", "Jagero", "childhood trauma"],
        "num_speakers": 2,
        "strict_known_speakers": True,
        "output_subdir": "Kenya/Onyango Otieno (Rixpoet)",
    },
    "onyango_my_voice_was_beaten_out_of_me_by_my_father_toxic_masculinity": {
        "youtube_url": "https://www.youtube.com/watch?v=qKTyoG1PXJc",
        "local_audio_path": "Kenya Audio Files/Onyango Otieno (Rixpoet)/My voice was beaten out of me by my father - Toxic masculinity [qKTyoG1PXJc].mp3",
        "primary_speaker_name": "Onyango Otieno (Rixpoet)",
        "known_speakers": [
            "Onyango Otieno (Rixpoet)",
            {"name": "Justus Nandwa", "aliases": ["Justus", "Justus Nandwa"]},
        ],
        "speaker_strategy": "interview_guest_host",
        "host_name": "Justus Nandwa",
        "guest_name": "Onyango Otieno (Rixpoet)",
        "content_format": "interview",
        "term_hints": ["Onyango Otieno", "Rixpoet", "Justus Nandwa", "toxic masculinity"],
        "num_speakers": 2,
        "strict_known_speakers": True,
        "output_subdir": "Kenya/Onyango Otieno (Rixpoet)",
    },
    "onyango_undoing_my_fathers_damage": {
        "youtube_url": "https://www.youtube.com/watch?v=BCDdf4fSpYg",
        "local_audio_path": "Kenya Audio Files/Onyango Otieno (Rixpoet)/Undoing My Fathers Damage - Onyango Otieno (Rix poet).mp3",
        "primary_speaker_name": "Onyango Otieno (Rixpoet)",
        "content_format": "interview",
        "term_hints": ["Onyango Otieno", "Rixpoet"],
        "min_speakers": 1,
        "max_speakers": 3,
        "output_subdir": "Kenya/Onyango Otieno (Rixpoet)",
    },
    "onyango_your_story_i_thought_having_a_lot_of_sex_would_cure_my_depression": {
        "youtube_url": "https://www.youtube.com/watch?v=NswcwAw_Pw4",
        "local_audio_path": "Kenya Audio Files/Onyango Otieno (Rixpoet)/Your Story_ I Thought Having A Lot Of Sex Would Cure My Depression.mp3",
        "primary_speaker_name": "Onyango Otieno (Rixpoet)",
        "speaker_strategy": "primary_plus_known_speakers",
        "known_speakers": [
            "Onyango Otieno (Rixpoet)",
            {"name": "Narrator", "aliases": ["Narrator"]},
        ],
        "content_format": "documentary",
        "term_hints": ["Onyango Otieno", "Rixpoet", "depression", "narrator"],
        "min_speakers": 2,
        "max_speakers": 4,
        "strict_known_speakers": True,
        "output_subdir": "Kenya/Onyango Otieno (Rixpoet)",
    },
    "philip_karanja_my_childhood_upbringing": {
        "youtube_url": "https://www.youtube.com/watch?v=orlXCgGMmyg",
        "local_audio_path": "Kenya Audio Files/Philip Karanja/1753. My Childhood Upbringing - Philip Karanja (@OfficialPhilKaranja) #cta101 [orlXCgGMmyg].mp3",
        "primary_speaker_name": "Philip Karanja",
        "known_speakers": [
            "Philip Karanja",
            {"name": "CTA Host", "aliases": ["CTA Host", "CTA 101 Host"]},
        ],
        "speaker_strategy": "interview_guest_host",
        "host_name": "CTA Host",
        "guest_name": "Philip Karanja",
        "content_format": "interview",
        "term_hints": ["Philip Karanja", "CTA Host", "CTA 101"],
        "num_speakers": 2,
        "strict_known_speakers": True,
        "output_subdir": "Kenya/Philip Karanja",
    },
    "philip_karanja_episode_1_a_girl_dad_on_a_mission": {
        "youtube_url": "https://www.youtube.com/watch?v=9lkDNN9BQOU",
        "local_audio_path": "Kenya Audio Files/Philip Karanja/Episode 1_ A Girl Dad on a Mission_Is my daughter really safe in this world_ [9lkDNN9BQOU].mp3",
        "primary_speaker_name": "Philip Karanja",
        "known_speakers": [
            "Philip Karanja",
            {"name": "Kambua Manundu", "aliases": ["Kambua", "Kambua Manundu"]},
        ],
        "speaker_strategy": "primary_plus_known_speakers",
        "content_format": "documentary",
        "term_hints": ["Philip Karanja", "Kambua Manundu", "Equality Now", "FGM", "femicide", "Kajiado"],
        "context_notes": "This is documentary-style audio with narration, interview clips, and news inserts. Do not collapse every voice into the host.",
        "min_speakers": 3,
        "max_speakers": 6,
        "output_subdir": "Kenya/Philip Karanja",
    },
    "philip_karanja_episode_2_a_girl_dad_on_a_mission": {
        "youtube_url": "https://www.youtube.com/watch?v=9lo5OZ-wdfg",
        "local_audio_path": "Kenya Audio Files/Philip Karanja/Episode 2_ A Girl Dad on a Mission_1 in 4 young women are married as children [9lo5OZ-wdfg].mp3",
        "primary_speaker_name": "Philip Karanja",
        "known_speakers": [
            "Philip Karanja",
            {"name": "Kambua Manundu", "aliases": ["Kambua", "Kambua Manundu"]},
        ],
        "speaker_strategy": "primary_plus_known_speakers",
        "content_format": "documentary",
        "term_hints": ["Philip Karanja", "Kambua Manundu", "FGM", "child marriage", "Kajiado", "Narok"],
        "context_notes": "This is documentary-style audio with narration, interview clips, and on-location speakers. Preserve sensitive wording if it is in the source.",
        "min_speakers": 3,
        "max_speakers": 6,
        "output_subdir": "Kenya/Philip Karanja",
    },
    "philip_karanja_season_finale_a_girl_dad_on_a_mission": {
        "youtube_url": "https://www.youtube.com/watch?v=Ai5zq5Nf8fA",
        "local_audio_path": "Kenya Audio Files/Philip Karanja/Season Finale_ A Girl Dad on a Mission_ Rape as Background Noise_ A Society Numbed by Violence [Ai5zq5Nf8fA].mp3",
        "primary_speaker_name": "Philip Karanja",
        "known_speakers": [
            "Philip Karanja",
            {"name": "Kambua Manundu", "aliases": ["Kambua", "Kambua Manundu"]},
        ],
        "speaker_strategy": "primary_plus_known_speakers",
        "content_format": "documentary",
        "term_hints": ["Philip Karanja", "Kambua Manundu", "rape", "gender-based violence", "femicide"],
        "context_notes": "This is documentary-style audio with narration, interview clips, and broadcast inserts. Preserve exact violence-related wording when present in the source.",
        "min_speakers": 3,
        "max_speakers": 6,
        "output_subdir": "Kenya/Philip Karanja",
    },
}

INFLUENCER_COLLECTIONS = {
    "Andrew Kibe": [
        "andrew_kibe_071_28_commandments_of_journeying_into_wealth_health_and_respect",
    ],
    "Onyango Otieno (Rixpoet)": [
        "onyango_narelate_mens_mental_health_workshop_nakuru_january_2023",
        "onyango_men_addiction_and_violence_the_story_of_our_childhood_trauma",
        "onyango_my_voice_was_beaten_out_of_me_by_my_father_toxic_masculinity",
        "onyango_undoing_my_fathers_damage",
        "onyango_your_story_i_thought_having_a_lot_of_sex_would_cure_my_depression",
    ],
    "Philip Karanja": [
        "philip_karanja_my_childhood_upbringing",
        "philip_karanja_episode_1_a_girl_dad_on_a_mission",
        "philip_karanja_episode_2_a_girl_dad_on_a_mission",
        "philip_karanja_season_finale_a_girl_dad_on_a_mission",
    ],
}


def rebuild_runtime_dirs(job_name: str, output_subdir: str = "") -> None:
    global OUTPUT_SUBDIR, TEMP_DIR, output_dir, temp_dir, artifacts_to_cleanup
    OUTPUT_SUBDIR = output_subdir or ""
    TEMP_DIR = f"temp/{job_name}"
    output_dir = (ROOT / OUTPUT_DIR).resolve()
    if OUTPUT_SUBDIR:
        output_dir = (output_dir / OUTPUT_SUBDIR).resolve()
    temp_dir = (ROOT / TEMP_DIR).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    temp_dir.mkdir(parents=True, exist_ok=True)
    artifacts_to_cleanup = []


def apply_video_job(job_name: str) -> dict:
    global YOUTUBE_URL, LOCAL_AUDIO_PATH, AUDIO_WAV_PATH, PRIMARY_SPEAKER_NAME, TRANSCRIPT_FILENAME
    global KNOWN_SPEAKERS, SPEAKER_STRATEGY, HOST_NAME, GUEST_NAME, SECONDARY_SPEAKER_NAME
    global NUM_SPEAKERS, MIN_SPEAKERS, MAX_SPEAKERS, STRICT_KNOWN_SPEAKERS
    global CONTENT_FORMAT, TERM_HINTS, CONTEXT_NOTES

    job = VIDEO_JOBS[job_name]
    YOUTUBE_URL = job.get("youtube_url", "")
    LOCAL_AUDIO_PATH = job.get("local_audio_path", "")
    AUDIO_WAV_PATH = job.get("audio_wav_path")
    PRIMARY_SPEAKER_NAME = job.get("primary_speaker_name")
    KNOWN_SPEAKERS = list(job.get("known_speakers", []))
    SPEAKER_STRATEGY = job.get("speaker_strategy", "default")
    HOST_NAME = job.get("host_name")
    GUEST_NAME = job.get("guest_name")
    SECONDARY_SPEAKER_NAME = job.get("secondary_speaker_name")
    NUM_SPEAKERS = job.get("num_speakers")
    MIN_SPEAKERS = job.get("min_speakers")
    MAX_SPEAKERS = job.get("max_speakers")
    STRICT_KNOWN_SPEAKERS = job.get("strict_known_speakers", False)
    CONTENT_FORMAT = job.get("content_format", "auto")
    TERM_HINTS = list(job.get("term_hints", []))
    CONTEXT_NOTES = job.get("context_notes", "")
    TRANSCRIPT_FILENAME = job.get("transcript_filename", "auto")
    rebuild_runtime_dirs(job_name, job.get("output_subdir", ""))
    print("Selected job:", job_name)
    print("YouTube URL:", YOUTUBE_URL or "not set")
    print("Local audio:", LOCAL_AUDIO_PATH or "not set")
    print("Output directory:", output_dir)
    print("Transcript filename mode:", TRANSCRIPT_FILENAME)
    print("Content format:", CONTENT_FORMAT)
    return job


def apply_custom_job(
    job_name: str,
    *,
    youtube_url: str = "",
    local_audio_path: str = "",
    audio_wav_path: Optional[str] = None,
    primary_speaker_name: Optional[str] = None,
    known_speakers: Optional[list[str]] = None,
    speaker_strategy: str = "default",
    host_name: Optional[str] = None,
    guest_name: Optional[str] = None,
    secondary_speaker_name: Optional[str] = None,
    content_format: str = "auto",
    term_hints: Optional[list[str]] = None,
    context_notes: str = "",
    num_speakers: Optional[int] = None,
    min_speakers: Optional[int] = None,
    max_speakers: Optional[int] = None,
    strict_known_speakers: bool = False,
    output_subdir: str = "",
    transcript_filename: str = "auto",
) -> dict:
    VIDEO_JOBS[job_name] = {
        "youtube_url": youtube_url,
        "local_audio_path": local_audio_path,
        "audio_wav_path": audio_wav_path,
        "primary_speaker_name": primary_speaker_name,
        "known_speakers": known_speakers or [],
        "speaker_strategy": speaker_strategy,
        "host_name": host_name,
        "guest_name": guest_name,
        "secondary_speaker_name": secondary_speaker_name,
        "content_format": content_format,
        "term_hints": term_hints or [],
        "context_notes": context_notes,
        "num_speakers": num_speakers,
        "min_speakers": min_speakers,
        "max_speakers": max_speakers,
        "strict_known_speakers": strict_known_speakers,
        "output_subdir": output_subdir,
        "transcript_filename": transcript_filename,
    }
    return apply_video_job(job_name)


def apply_influencer_collection(collection_name: str) -> list[str]:
    job_names = INFLUENCER_COLLECTIONS[collection_name]
    print("Influencer:", collection_name)
    print("Jobs in this influencer set:")
    for job_name in job_names:
        job = VIDEO_JOBS[job_name]
        print("-", job_name)
        print("  audio:", job.get("local_audio_path"))
        print("  output:", (Path(OUTPUT_DIR) / job.get("output_subdir", "")).as_posix())
    print("Run one apply_video_job(...) line from this cell next.")
    return job_names


print("Available Nigeria jobs:")
for job_name in [
    "face_it_like_a_man",
    "faith_after_a_fall",
    "final_say_faith",
    "my_story_journey_through_hope_and_faith",
    "the_prison_of_pornography",
]:
    print("-", job_name)

print("\nAvailable Kenya influencer collections:")
for collection_name in INFLUENCER_COLLECTIONS:
    print("-", collection_name)


# Nigeria presets


In [ ]:
apply_video_job("face_it_like_a_man")


In [ ]:
apply_video_job("faith_after_a_fall")


In [ ]:
apply_video_job("final_say_faith")


In [ ]:
apply_video_job("my_story_journey_through_hope_and_faith")


In [ ]:
apply_video_job("the_prison_of_pornography")


# Kenya presets


In [ ]:
apply_influencer_collection("Andrew Kibe")
# Uncomment the job you want to run from this influencer set.
# apply_video_job("andrew_kibe_071_28_commandments_of_journeying_into_wealth_health_and_respect")


In [ ]:
apply_influencer_collection("Onyango Otieno (Rixpoet)")
# Uncomment one job at a time from this influencer set.
# apply_video_job("onyango_narelate_mens_mental_health_workshop_nakuru_january_2023")
# apply_video_job("onyango_men_addiction_and_violence_the_story_of_our_childhood_trauma")
# apply_video_job("onyango_my_voice_was_beaten_out_of_me_by_my_father_toxic_masculinity")
# apply_video_job("onyango_undoing_my_fathers_damage")
# apply_video_job("onyango_your_story_i_thought_having_a_lot_of_sex_would_cure_my_depression")


In [ ]:
apply_influencer_collection("Philip Karanja")
# Uncomment one job at a time from this influencer set.
# apply_video_job("philip_karanja_my_childhood_upbringing")
# apply_video_job("philip_karanja_episode_1_a_girl_dad_on_a_mission")
# apply_video_job("philip_karanja_episode_2_a_girl_dad_on_a_mission")
# apply_video_job("philip_karanja_season_finale_a_girl_dad_on_a_mission")


In [ ]:
# Copy this cell to add a new audio file or YouTube source.
# apply_custom_job(
#     "new_audio_example",
#     youtube_url="https://www.youtube.com/watch?v=VIDEO_ID",
#     local_audio_path="Kenya Audio Files/Influencer Name/your audio file.mp3",
#     primary_speaker_name="Influencer Name",
#     content_format="interview",
#     output_subdir="Kenya/Influencer Name",
# )


# YouTube download


In [ ]:
require_binary("ffmpeg")

from yt_dlp import YoutubeDL


def build_metadata_from_local_file(local_path: str) -> dict:
    path = Path(local_path).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(f"Local audio file does not exist: {path}")

    title = clean_text(re.sub(r"\s*\[[^\]]+\]$", "", path.stem).replace("_", " "))
    return {
        "audio_path": str(path),
        "title": title,
        "uploader": None,
        "channel": None,
        "view_count": None,
        "like_count": None,
        "comment_count": None,
        "description": None,
        "duration": None,
        "tags": [],
        "webpage_url": None,
    }


def fetch_youtube_metadata(url: str) -> dict:
    try:
        with YoutubeDL({"quiet": True, "skip_download": True, "noplaylist": True}) as ydl:
            info = ydl.extract_info(url, download=False)
        return {
            "title": info.get("title"),
            "uploader": info.get("uploader"),
            "channel": info.get("channel"),
            "view_count": info.get("view_count"),
            "like_count": info.get("like_count"),
            "comment_count": info.get("comment_count"),
            "description": info.get("description"),
            "duration": info.get("duration"),
            "tags": info.get("tags") or [],
            "webpage_url": info.get("webpage_url") or url,
        }
    except Exception as exc:
        log.warning("Could not fetch YouTube metadata for %s: %s", url, exc)
        return {"webpage_url": url}


def merge_metadata(local_metadata: dict, remote_metadata: Optional[dict] = None) -> dict:
    merged = dict(local_metadata)
    for key, value in (remote_metadata or {}).items():
        if key == "audio_path":
            continue
        if value not in (None, "", []):
            merged[key] = value
    return merged


def download_youtube_audio(url: str, out_dir: str) -> dict:
    out_dir_path = Path(out_dir).expanduser().resolve()
    out_dir_path.mkdir(parents=True, exist_ok=True)

    ydl_opts = {
        "format": "bestaudio/best",
        "noplaylist": True,
        "quiet": False,
        "restrictfilenames": False,
        "outtmpl": str(out_dir_path / "%(title).200B [%(id)s].%(ext)s"),
    }

    try:
        with YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            downloaded_path = None

            requested_downloads = info.get("requested_downloads") or []
            if requested_downloads:
                downloaded_path = requested_downloads[0].get("filepath")

            if not downloaded_path:
                downloaded_path = ydl.prepare_filename(info)

            audio_path = Path(downloaded_path).expanduser().resolve()
            if not audio_path.exists():
                raise FileNotFoundError(f"Downloaded audio file was not found: {audio_path}")

            return {
                "audio_path": str(audio_path),
                "title": info.get("title") or audio_path.stem,
                "uploader": info.get("uploader"),
                "channel": info.get("channel"),
                "view_count": info.get("view_count"),
                "like_count": info.get("like_count"),
                "comment_count": info.get("comment_count"),
                "description": info.get("description"),
                "duration": info.get("duration"),
                "tags": info.get("tags") or [],
                "webpage_url": info.get("webpage_url") or url,
            }
    except Exception as exc:
        raise RuntimeError(f"yt-dlp download failed: {exc}") from exc


metadata_source = "local"

if AUDIO_WAV_PATH:
    metadata = build_metadata_from_local_file(AUDIO_WAV_PATH)
    source_audio_path = Path(metadata["audio_path"]).resolve()
    if YOUTUBE_URL:
        metadata = merge_metadata(metadata, fetch_youtube_metadata(YOUTUBE_URL))
        metadata_source = "local audio + YouTube metadata"
elif LOCAL_AUDIO_PATH:
    metadata = build_metadata_from_local_file(LOCAL_AUDIO_PATH)
    source_audio_path = Path(metadata["audio_path"]).resolve()
    if YOUTUBE_URL:
        metadata = merge_metadata(metadata, fetch_youtube_metadata(YOUTUBE_URL))
        metadata_source = "local audio + YouTube metadata"
elif YOUTUBE_URL:
    metadata = download_youtube_audio(YOUTUBE_URL, str(temp_dir))
    source_audio_path = Path(metadata["audio_path"]).resolve()
    artifacts_to_cleanup.append(source_audio_path)
    metadata_source = "downloaded from YouTube"
else:
    raise ValueError("Set YOUTUBE_URL or LOCAL_AUDIO_PATH or AUDIO_WAV_PATH before running this cell.")

primary_speaker_guess = infer_primary_speaker_name(metadata, PRIMARY_SPEAKER_NAME)
metadata["primary_speaker"] = primary_speaker_guess
metadata["known_speakers"] = list(KNOWN_SPEAKERS)
metadata["speaker_strategy"] = SPEAKER_STRATEGY
metadata["host_name"] = HOST_NAME
metadata["guest_name"] = GUEST_NAME
metadata["secondary_speaker_name"] = SECONDARY_SPEAKER_NAME
metadata["strict_known_speakers"] = STRICT_KNOWN_SPEAKERS
metadata["content_format"] = CONTENT_FORMAT
metadata["term_hints"] = stable_dedupe(list(TERM_HINTS) + list(metadata.get("tags") or [])[:12])
metadata["context_notes"] = clean_text(CONTEXT_NOTES)

print("Metadata source:", metadata_source)
print("Source audio path:", source_audio_path)
print("Title:", metadata.get("title"))
print("Primary speaker guess:", metadata.get("primary_speaker"))
print("Content format hint:", metadata.get("content_format"))
print("Stats line:", build_stats_line(metadata) or "not available")


# Pipeline preflight


In [ ]:
def resolve_api_key() -> str:
    key = (GEMINI_API_KEY or GOOGLE_API_KEY or "").strip()
    return key


def ensure_runtime_is_ready() -> None:
    if not LOCAL_AUDIO_PATH and not AUDIO_WAV_PATH and not YOUTUBE_URL:
        raise RuntimeError("Select a preset or set LOCAL_AUDIO_PATH / AUDIO_WAV_PATH / YOUTUBE_URL first.")
    if USE_GEMINI_VIDEO_CONTEXT or USE_GEMINI_REFINEMENT:
        api_key = resolve_api_key()
        if not api_key:
            raise RuntimeError(
                "Gemini refinement is enabled but no GEMINI_API_KEY or GOOGLE_API_KEY is set in .env."
            )
    if LOCAL_AUDIO_PATH:
        audio_path = Path(LOCAL_AUDIO_PATH).expanduser().resolve()
        if not audio_path.exists():
            raise FileNotFoundError(f"Configured local audio file does not exist: {audio_path}")
    if AUDIO_WAV_PATH:
        wav_source = Path(AUDIO_WAV_PATH).expanduser().resolve()
        if not wav_source.exists():
            raise FileNotFoundError(f"Configured AUDIO_WAV_PATH does not exist: {wav_source}")
    if not HF_TOKEN:
        raise RuntimeError("HF_TOKEN is required for pyannote diarization.")


def ping_gemini_text() -> None:
    api_key = resolve_api_key()
    if not api_key:
        return

    from google import genai
    from google.genai import types

    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents="Reply with exactly OK",
        config=types.GenerateContentConfig(
            temperature=0,
            maxOutputTokens=16,
        ),
    )
    text = extract_gemini_text(response)
    if text.strip().upper() != "OK":
        raise RuntimeError(f"Gemini preflight returned unexpected text: {text!r}")


ensure_runtime_is_ready()
ping_gemini_text()
print("Preflight passed.")
print("Selected audio:", LOCAL_AUDIO_PATH or AUDIO_WAV_PATH or YOUTUBE_URL)
print("Gemini model:", GEMINI_MODEL)
print("Gemini key present:", bool(resolve_api_key()))


# Video context grounding


In [ ]:
class VideoContextPayload(TypedDict, total=False):
    content_format: str
    likely_named_speakers: list[str]
    term_hints: list[str]
    language_notes: str
    context_notes: list[str]
    speaker_count_hint: int


def parse_json_response(text: str, *, fallback: Optional[dict] = None) -> dict:
    text = (text or "").strip()
    if not text:
        return fallback or {}
    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except Exception:
                pass
    return fallback or {}


def build_gemini_safety_settings():
    if not GEMINI_ENABLE_SAFETY_OVERRIDE:
        return None

    try:
        from google.genai import types
    except Exception:
        return None

    categories = [
        types.HarmCategory.HARM_CATEGORY_HARASSMENT,
        types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
        types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
        types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
    ]
    return [
        types.SafetySetting(category=category, threshold=types.HarmBlockThreshold.BLOCK_NONE)
        for category in categories
    ]


def build_video_context_prompt(metadata: dict) -> str:
    description = clean_text((metadata.get("description") or "")[:4000])
    tags = ", ".join((metadata.get("tags") or [])[:20])
    return dedent(
        f'''
        Ground a transcript-repair pipeline for this YouTube video using the metadata and description below.

        Return JSON only with this schema:
        {{
          "content_format": "sermon | interview | documentary | workshop | conversation_testimony | panel | default",
          "likely_named_speakers": ["name", "name"],
          "term_hints": ["term", "term"],
          "language_notes": "short note",
          "context_notes": ["short note", "short note"],
          "speaker_count_hint": 0
        }}

        Constraints:
        - Be conservative. Do not invent names unless strongly supported by the metadata, title, or description.
        - Prefer exact names that appear in the title, channel framing, description, or repeated context clues.
        - For documentary-style videos, mention that narration, interviews, and broadcast inserts may be present.
        - Focus on information useful for speaker attribution and ASR repair.

        Video metadata:
        - URL: {metadata.get("webpage_url")}
        - title: {metadata.get("title")}
        - uploader: {metadata.get("uploader")}
        - channel: {metadata.get("channel")}
        - duration_seconds: {metadata.get("duration")}
        - content_format_hint: {metadata.get("content_format")}
        - primary_speaker_guess: {metadata.get("primary_speaker")}
        - local_term_hints: {metadata.get("term_hints")}
        - local_notes: {metadata.get("context_notes")}
        - tags: {tags}

        Description excerpt:
        {description}
        '''
    ).strip()


def extract_video_context_with_gemini(metadata: dict) -> dict:
    if not USE_GEMINI_VIDEO_CONTEXT:
        return {}
    api_key = resolve_api_key()
    if not api_key:
        raise RuntimeError("Gemini video grounding is enabled but no API key was found.")

    try:
        from google import genai
        from google.genai import types
    except Exception as exc:
        raise RuntimeError(f"google.genai is unavailable: {exc}") from exc

    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model=GEMINI_CONTEXT_MODEL,
        contents=build_video_context_prompt(metadata),
        config=types.GenerateContentConfig(
            temperature=0,
            maxOutputTokens=2048,
            responseMimeType="application/json",
            safetySettings=build_gemini_safety_settings(),
        ),
    )

    text = extract_gemini_text(response)
    if not text:
        raise RuntimeError("Gemini returned an empty response for video grounding.")

    parsed = parse_json_response(text, fallback={})
    context: VideoContextPayload = {
        "content_format": clean_text(str(parsed.get("content_format") or "")).lower() or metadata.get("content_format") or "default",
        "likely_named_speakers": stable_dedupe([str(item) for item in parsed.get("likely_named_speakers") or []]),
        "term_hints": stable_dedupe([str(item) for item in parsed.get("term_hints") or []]),
        "language_notes": clean_text(str(parsed.get("language_notes") or "")),
        "context_notes": stable_dedupe([str(item) for item in parsed.get("context_notes") or []]),
        "speaker_count_hint": int(parsed.get("speaker_count_hint") or 0) if str(parsed.get("speaker_count_hint") or "").isdigit() else 0,
    }
    return {key: value for key, value in context.items() if value not in ("", [], 0, None)}


video_context = extract_video_context_with_gemini(metadata)
metadata["video_context"] = video_context
if video_context.get("content_format") and metadata.get("content_format") in ("", None, "auto", "default"):
    metadata["content_format"] = video_context["content_format"]
metadata["term_hints"] = stable_dedupe(list(metadata.get("term_hints") or []) + list(video_context.get("term_hints") or []))
metadata["video_context_notes"] = video_context.get("context_notes") or []

print("Video context enabled:", USE_GEMINI_VIDEO_CONTEXT and bool(resolve_api_key()))
print("Video context summary:", json.dumps(video_context, ensure_ascii=False, indent=2) if video_context else "not available")


# Audio conversion


In [ ]:
def convert_to_wav_16k_mono(input_path: str, output_path: str) -> str:
    require_binary('ffmpeg')
    input_path = str(Path(input_path).expanduser().resolve())
    output_path = str(Path(output_path).expanduser().resolve())

    command = [
        'ffmpeg',
        '-y',
        '-i', input_path,
        '-ac', '1',
        '-ar', '16000',
        '-c:a', 'pcm_s16le',
        output_path,
    ]

    try:
        run_command(command)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(exc.stderr or exc.stdout or 'ffmpeg conversion failed') from exc

    output = Path(output_path)
    if not output.exists():
        raise FileNotFoundError(f'Converted WAV file was not created: {output}')
    return str(output)


def get_audio_duration(path: str) -> float:
    require_binary('ffprobe')
    command = [
        'ffprobe',
        '-v', 'error',
        '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        str(Path(path).expanduser().resolve()),
    ]
    try:
        completed = run_command(command)
        return float((completed.stdout or '').strip())
    except Exception as exc:
        raise RuntimeError(f'Could not determine audio duration for {path}: {exc}') from exc


if AUDIO_WAV_PATH:
    wav_path = Path(AUDIO_WAV_PATH).expanduser().resolve()
    if not wav_path.exists():
        raise FileNotFoundError(f'WAV input does not exist: {wav_path}')
else:
    wav_name = safe_slug(metadata.get('title') or source_audio_path.stem, fallback='audio') + '_16khz_mono.wav'
    wav_path = (temp_dir / wav_name).resolve()
    convert_to_wav_16k_mono(str(source_audio_path), str(wav_path))
    if wav_path not in artifacts_to_cleanup:
        artifacts_to_cleanup.append(wav_path)

audio_duration = get_audio_duration(str(wav_path))
print('Prepared WAV path:', wav_path)
print(f'Audio duration: {audio_duration:.2f} seconds')


# ASR transcription


In [ ]:
def normalize_word_entries(raw_words: list[dict], *, source: str) -> list[dict]:
    normalized = []
    for item in raw_words:
        word = item.get("word")
        start = item.get("start")
        end = item.get("end")
        if word is None or start is None or end is None:
            continue
        normalized.append(
            {
                "word": str(word),
                "start": float(start),
                "end": float(end),
                "source": source,
            }
        )
    return normalized


def extract_term_hints(metadata: dict) -> list[str]:
    hints = []
    hints.extend(metadata.get("term_hints") or [])
    hints.extend(metadata.get("video_context", {}).get("likely_named_speakers") or [])
    if metadata.get("primary_speaker"):
        hints.append(metadata["primary_speaker"])
    if metadata.get("host_name"):
        hints.append(metadata["host_name"])
    if metadata.get("guest_name"):
        hints.append(metadata["guest_name"])
    if metadata.get("secondary_speaker_name"):
        hints.append(metadata["secondary_speaker_name"])
    for speaker in metadata.get("known_speakers") or []:
        if isinstance(speaker, str):
            hints.append(speaker)
        elif isinstance(speaker, dict):
            hints.append(speaker.get("name") or "")
            hints.extend(speaker.get("aliases") or [])
    return stable_dedupe(hints)


def build_asr_initial_prompt(metadata: dict) -> str:
    hints = extract_term_hints(metadata)
    language_notes = metadata.get("video_context", {}).get("language_notes") or ""
    content_format = metadata.get("content_format") or "default"
    parts = [
        "Create a faithful transcript with proper names preserved.",
        f"Content format: {content_format}.",
    ]
    if language_notes:
        parts.append(language_notes)
    if hints:
        parts.append("Important names and terms: " + ", ".join(hints[:40]) + ".")
    return clean_text(" ".join(parts))[:1200]


def normalize_mlx_result(result: Any, *, model_name: str) -> dict:
    if not isinstance(result, dict):
        raise RuntimeError("MLX Whisper result was not a dict-like object.")

    segments = result.get("segments") or []
    raw_words = []
    segment_texts = []
    for segment in segments:
        seg_text = segment.get("text") or ""
        if seg_text:
            segment_texts.append(seg_text.strip())
        for word in segment.get("words") or []:
            raw_words.append(
                {
                    "word": word.get("word") or word.get("text"),
                    "start": word.get("start"),
                    "end": word.get("end"),
                }
            )

    words = normalize_word_entries(raw_words, source="mlx_whisper")
    if not words:
        raise RuntimeError(
            "MLX Whisper did not return word-level timestamps. Use faster-whisper fallback or a different MLX build."
        )

    return {
        "backend": "mlx_whisper",
        "model_name": model_name,
        "full_text": clean_text(result.get("text") or " ".join(segment_texts)),
        "language": result.get("language"),
        "words": words,
    }


def transcribe_with_mlx_whisper(audio_path: str, model_name: str, *, initial_prompt: str = "") -> dict:
    try:
        import mlx_whisper
    except Exception as exc:
        raise RuntimeError("mlx_whisper is not installed in this environment.") from exc

    candidates = []
    mapped = {
        "large-v3": "mlx-community/whisper-large-v3-mlx",
        "large-v3-turbo": "mlx-community/whisper-large-v3-turbo",
        "turbo": "mlx-community/whisper-large-v3-turbo",
    }.get(model_name)
    for candidate in [model_name, mapped]:
        if candidate and candidate not in candidates:
            candidates.append(candidate)

    last_error = None
    for candidate in candidates:
        base_kwargs = {
            "word_timestamps": True,
            "condition_on_previous_text": False,
            "compression_ratio_threshold": 2.0,
            "hallucination_silence_threshold": 1.5,
        }
        if initial_prompt:
            base_kwargs["initial_prompt"] = initial_prompt

        try:
            result = mlx_whisper.transcribe(
                str(audio_path),
                path_or_hf_repo=candidate,
                **base_kwargs,
            )
            return normalize_mlx_result(result, model_name=candidate)
        except TypeError:
            try:
                result = mlx_whisper.transcribe(
                    str(audio_path),
                    model=candidate,
                    **base_kwargs,
                )
                return normalize_mlx_result(result, model_name=candidate)
            except Exception as inner_exc:
                last_error = inner_exc
        except Exception as exc:
            last_error = exc

    raise RuntimeError(f"MLX Whisper transcription failed: {last_error}")


def transcribe_with_faster_whisper(audio_path: str, model_name: str, *, initial_prompt: str = "") -> dict:
    try:
        from faster_whisper import WhisperModel
    except Exception as exc:
        raise RuntimeError("faster_whisper is not installed in this environment.") from exc

    model = WhisperModel(model_name, device="cpu", compute_type="int8")
    segments, info = model.transcribe(
        str(audio_path),
        beam_size=5,
        word_timestamps=True,
        vad_filter=True,
        condition_on_previous_text=False,
        initial_prompt=initial_prompt or None,
    )
    segments = list(segments)

    words = []
    full_text_parts = []
    for segment in segments:
        seg_text = getattr(segment, "text", "") or ""
        if seg_text:
            full_text_parts.append(seg_text.strip())
        for word in getattr(segment, "words", None) or []:
            if word.start is None or word.end is None:
                continue
            words.append(
                {
                    "word": word.word,
                    "start": float(word.start),
                    "end": float(word.end),
                    "source": "faster_whisper",
                }
            )

    if not words:
        raise RuntimeError("faster-whisper did not return word-level timestamps.")

    return {
        "backend": "faster_whisper",
        "model_name": model_name,
        "full_text": clean_text(" ".join(full_text_parts)),
        "language": getattr(info, "language", None),
        "words": words,
    }


def transcribe_audio(audio_path: str, model_name: str, backend: str = "auto", *, initial_prompt: str = "") -> dict:
    errors = []

    if backend in ("auto", "mlx_whisper"):
        try:
            result = transcribe_with_mlx_whisper(audio_path, model_name, initial_prompt=initial_prompt)
            print(f"ASR backend: {result['backend']} ({result['model_name']})")
            return result
        except Exception as exc:
            errors.append(f"mlx_whisper: {exc}")
            if backend == "mlx_whisper":
                raise
            log.warning("MLX Whisper unavailable or failed. Falling back to faster-whisper. Reason: %s", exc)

    result = transcribe_with_faster_whisper(audio_path, model_name, initial_prompt=initial_prompt)
    print(f"ASR backend: {result['backend']} ({result['model_name']})")
    if errors:
        print("Fallback notes:")
        for item in errors:
            print(" -", item)
    return result


asr_initial_prompt = build_asr_initial_prompt(metadata)
print("ASR initial prompt:", asr_initial_prompt[:500])

asr_result = transcribe_audio(str(wav_path), WHISPER_MODEL, backend=ASR_BACKEND, initial_prompt=asr_initial_prompt)
words = asr_result["words"]
metadata["asr_backend"] = asr_result["backend"]
metadata["asr_model"] = asr_result["model_name"]
metadata["language"] = asr_result.get("language")
metadata["asr_initial_prompt"] = asr_initial_prompt

print("Detected language:", metadata.get("language"))
print("Word count with timestamps:", len(words))
print("Transcript preview:", clean_text(asr_result["full_text"][:300]))


# Speaker diarization


In [ ]:
def derive_diarization_constraints(metadata: dict) -> dict:
    if NUM_SPEAKERS is not None:
        return {"num_speakers": int(NUM_SPEAKERS)}
    if MIN_SPEAKERS is not None or MAX_SPEAKERS is not None:
        result = {}
        if MIN_SPEAKERS is not None:
            result["min_speakers"] = int(MIN_SPEAKERS)
        if MAX_SPEAKERS is not None:
            result["max_speakers"] = int(MAX_SPEAKERS)
        return result

    content_format = (metadata.get("content_format") or metadata.get("video_context", {}).get("content_format") or "default").lower()
    explicit_names = []
    for speaker in metadata.get("known_speakers") or []:
        if isinstance(speaker, str):
            explicit_names.append(speaker)
        elif isinstance(speaker, dict) and speaker.get("name"):
            explicit_names.append(speaker["name"])

    if content_format in {"interview", "conversation_testimony"} and len(stable_dedupe(explicit_names)) >= 2:
        return {"num_speakers": 2}
    if content_format == "sermon":
        return {"min_speakers": 1, "max_speakers": 3}
    if content_format == "documentary":
        return {"min_speakers": 3, "max_speakers": 6}
    if content_format == "workshop":
        return {"min_speakers": 2, "max_speakers": 8}
    if content_format in {"interview", "conversation_testimony"}:
        return {"min_speakers": 2, "max_speakers": 3}
    return {}


def diarize_with_pyannote(
    audio_path: str,
    hf_token: str,
    *,
    num_speakers: Optional[int] = None,
    min_speakers: Optional[int] = None,
    max_speakers: Optional[int] = None,
) -> list[dict]:
    if not hf_token:
        raise ValueError(
            "HF_TOKEN is required for pyannote.audio diarization. "
            "You also need to accept the model terms for "
            "pyannote/speaker-diarization-community-1."
        )

    try:
        from pyannote.audio import Pipeline
        import torch
    except Exception as exc:
        raise RuntimeError("pyannote.audio is not installed in this environment.") from exc

    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-community-1",
        token=hf_token,
    )

    device = torch.device("cpu")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        device = torch.device("mps")
    try:
        pipeline.to(device)
    except Exception as exc:
        if device.type != "cpu":
            log.warning("Could not move diarization pipeline to %s: %s. Falling back to CPU.", device, exc)
            device = torch.device("cpu")
            pipeline.to(device)
        else:
            raise

    diarization_kwargs = {}
    if num_speakers is not None:
        diarization_kwargs["num_speakers"] = int(num_speakers)
    else:
        if min_speakers is not None:
            diarization_kwargs["min_speakers"] = int(min_speakers)
        if max_speakers is not None:
            diarization_kwargs["max_speakers"] = int(max_speakers)

    diarization = pipeline(str(audio_path), **diarization_kwargs)
    annotation = getattr(diarization, "exclusive_speaker_diarization", None)
    if annotation is None:
        annotation = getattr(diarization, "speaker_diarization", diarization)

    label_map = {}
    normalized = []
    for turn, _, speaker in annotation.itertracks(yield_label=True):
        if speaker not in label_map:
            label_map[speaker] = f"Speaker {len(label_map) + 1}"
        normalized.append(
            {
                "speaker": label_map[speaker],
                "start": round(float(turn.start), 3),
                "end": round(float(turn.end), 3),
            }
        )

    if not normalized:
        raise RuntimeError("Diarization produced no speaker segments.")

    return normalized


constraints = derive_diarization_constraints(metadata)
metadata["diarization_constraints"] = constraints
print("Diarization constraints:", constraints or "none")

speaker_segments = diarize_with_pyannote(
    str(wav_path),
    HF_TOKEN,
    num_speakers=constraints.get("num_speakers"),
    min_speakers=constraints.get("min_speakers"),
    max_speakers=constraints.get("max_speakers"),
)

print("Diarization segments:", len(speaker_segments))
print("First segments:", speaker_segments[:5])


# Merge transcription + diarization


In [ ]:
def assign_speakers_to_words(words: list[dict], speaker_segments: list[dict], tolerance: float = 0.75) -> list[dict]:
    if not speaker_segments:
        return [{**word, 'speaker': 'Unknown'} for word in words]

    ordered_segments = sorted(speaker_segments, key=lambda item: (item['start'], item['end']))
    assigned = []

    for word in words:
        start = word.get('start')
        end = word.get('end')
        if start is None or end is None:
            assigned.append({**word, 'speaker': 'Unknown'})
            continue

        midpoint = (float(start) + float(end)) / 2.0
        speaker = None

        for segment in ordered_segments:
            if segment['start'] <= midpoint <= segment['end']:
                speaker = segment['speaker']
                break

        if speaker is None:
            nearest_segment = None
            nearest_distance = None
            for segment in ordered_segments:
                if midpoint < segment['start']:
                    distance = segment['start'] - midpoint
                elif midpoint > segment['end']:
                    distance = midpoint - segment['end']
                else:
                    distance = 0.0

                if nearest_distance is None or distance < nearest_distance:
                    nearest_distance = distance
                    nearest_segment = segment

            if nearest_segment is not None and nearest_distance is not None and nearest_distance <= tolerance:
                speaker = nearest_segment['speaker']
            else:
                speaker = 'Unknown'

        assigned.append({**word, 'speaker': speaker})

    return assigned


def join_word_tokens(tokens: list[str]) -> str:
    pieces = []
    for token in tokens:
        token = token or ''
        if not token:
            continue
        if not pieces:
            pieces.append(token.lstrip())
            continue
        if token.startswith(' '):
            pieces.append(token)
        elif re.match(r'^[,.;:!?%\)\]\}]', token):
            pieces.append(token)
        elif token.startswith(("'", '’')):
            pieces.append(token)
        else:
            pieces.append(' ' + token)
    return clean_text(''.join(pieces))


def collapse_words_into_turns(words_with_speakers: list[dict], max_gap: float = 0.9) -> list[dict]:
    turns = []
    current = None

    for item in words_with_speakers:
        token = item.get('word')
        start = item.get('start')
        end = item.get('end')
        speaker = item.get('speaker') or 'Unknown'
        if token is None or start is None or end is None:
            continue

        if current is None:
            current = {
                'speaker': speaker,
                'start': float(start),
                'end': float(end),
                'tokens': [str(token)],
            }
            continue

        gap = float(start) - float(current['end'])
        speaker_changed = speaker != current['speaker']
        gap_too_large = gap > max_gap

        if speaker_changed or gap_too_large:
            text = join_word_tokens(current['tokens'])
            if text:
                turns.append(
                    {
                        'speaker': current['speaker'],
                        'start': round(float(current['start']), 3),
                        'end': round(float(current['end']), 3),
                        'text': text,
                    }
                )
            current = {
                'speaker': speaker,
                'start': float(start),
                'end': float(end),
                'tokens': [str(token)],
            }
        else:
            current['tokens'].append(str(token))
            current['end'] = float(end)

    if current is not None:
        text = join_word_tokens(current['tokens'])
        if text:
            turns.append(
                {
                    'speaker': current['speaker'],
                    'start': round(float(current['start']), 3),
                    'end': round(float(current['end']), 3),
                    'text': text,
                }
            )

    return turns


def absorb_micro_fragments(turns: list[dict], max_words: int = 3) -> list[dict]:
    """Absorb tiny turns (<=max_words) into neighboring turns when surrounded by same speaker."""
    if len(turns) < 3:
        return turns

    # Pass 1: Absorb tiny turns sandwiched between same-speaker turns
    absorbed = list(turns)
    changed = True
    while changed:
        changed = False
        new_turns = []
        i = 0
        while i < len(absorbed):
            if i == 0 or i == len(absorbed) - 1:
                new_turns.append(absorbed[i])
                i += 1
                continue

            current = absorbed[i]
            prev_turn = new_turns[-1] if new_turns else None
            next_turn = absorbed[i + 1]
            word_count = len(re.findall(r"[A-Za-z0-9']+", clean_text(current.get('text', ''))))

            # If tiny turn is sandwiched between same speaker, absorb into previous
            if (prev_turn and word_count <= max_words
                    and prev_turn['speaker'] == next_turn['speaker']
                    and current['speaker'] != prev_turn['speaker']):
                prev_turn['text'] = clean_text(prev_turn['text'] + ' ' + current['text'])
                prev_turn['end'] = max(float(prev_turn['end']), float(current['end']))
                changed = True
                i += 1
                continue

            new_turns.append(current)
            i += 1
        absorbed = new_turns

    # Pass 2: Merge consecutive same-speaker turns
    merged = []
    for turn in absorbed:
        if merged and merged[-1]['speaker'] == turn['speaker']:
            merged[-1]['text'] = clean_text(merged[-1]['text'] + ' ' + turn['text'])
            merged[-1]['end'] = max(float(merged[-1]['end']), float(turn['end']))
        else:
            merged.append(dict(turn))

    return merged


words_with_speakers = assign_speakers_to_words(words, speaker_segments, tolerance=0.75)
turns = collapse_words_into_turns(words_with_speakers, max_gap=0.9)
turns = absorb_micro_fragments(turns, max_words=3)

print('Merged words with speakers:', len(words_with_speakers))
print('Initial turns:', len(turns))
print('Turn preview:', turns[:3])


# Speaker relabeling


In [ ]:
# transcript_output_utils guards
from scripts.transcript_output_utils import (
    extract_jsonish_turns as repair_extract_jsonish_turns,
    is_jsonish_transcript_text as repair_is_jsonish_transcript_text,
    parse_transcript_style_lines as repair_parse_transcript_style_lines,
    sanitize_output_turns as repair_sanitize_output_turns,
)


def parse_transcript_style_lines(text: str) -> list[dict]:
    return repair_parse_transcript_style_lines(text)


def fallback_refined_turns(text: str, chunk_turns: list[dict]) -> list[dict]:
    if repair_is_jsonish_transcript_text(text):
        repaired = repair_extract_jsonish_turns(text)
        if repaired:
            return repaired
    line_parsed = repair_parse_transcript_style_lines(text)
    if line_parsed:
        return line_parsed
    return [
        {"speaker": turn.get("speaker") or "Unknown", "text": clean_text(turn.get("text", ""))}
        for turn in chunk_turns
        if clean_text(turn.get("text", ""))
    ]


In [ ]:
def is_audience_like_turn(text: str) -> bool:
    cleaned = clean_text(text).strip('"\'').lower()
    if not cleaned:
        return False
    short_patterns = {
        "amen",
        "amen!",
        "proceed",
        "proceed!",
        "yes",
        "yes sir",
        "hallelujah",
        "thank you jesus",
        "praise the lord",
        "wow",
        "stop being out",
    }
    return len(cleaned.split()) <= 5 and cleaned in short_patterns


def should_merge_turn_pair(previous: dict, current: dict, *, content_format: str = "default", max_gap: float = 1.25) -> bool:
    if previous["speaker"] != current["speaker"]:
        return False
    if current["start"] - previous["end"] > max_gap:
        return False
    prev_text = clean_text(previous.get("text", ""))
    curr_text = clean_text(current.get("text", ""))
    if not prev_text or not curr_text:
        return False
    short_previous = len(prev_text) <= 160
    short_current = len(curr_text) <= 160
    prev_ends_hard = bool(re.search(r"[.!?\"']$", prev_text))
    curr_starts_continuation = bool(re.match(r"^(and|but|so|because|then|that|which|who|where|when|while|if|or)\b", curr_text, re.I))
    if short_previous or short_current:
        return True
    if not prev_ends_hard or curr_starts_continuation:
        return True
    if content_format in {"interview", "conversation_testimony"} and len(prev_text) < 300 and len(curr_text) < 300:
        return True
    return False


def merge_adjacent_turns(turns: list[dict], max_gap: float = 1.25, *, content_format: str = "default") -> list[dict]:
    merged = []
    for turn in turns:
        text = clean_text(turn.get("text", ""))
        if not text:
            continue
        current = {
            "speaker": clean_text(turn.get("speaker") or "Unknown") or "Unknown",
            "start": float(turn.get("start", 0.0)),
            "end": float(turn.get("end", 0.0)),
            "text": text,
        }
        if merged and should_merge_turn_pair(merged[-1], current, content_format=content_format, max_gap=max_gap):
            merged[-1]["end"] = max(merged[-1]["end"], current["end"])
            merged[-1]["text"] = clean_text(merged[-1]["text"] + " " + current["text"])
        else:
            merged.append(current)
    return merged


def split_long_turn_text(text: str, target_chars: int = 950, hard_limit: int = 1800) -> list[str]:
    text = clean_text(text)
    if len(text) <= hard_limit:
        return [text]
    sentences = [segment.strip() for segment in re.split(r"(?<=[.!?])\s+", text) if segment.strip()]
    if len(sentences) <= 1:
        words = text.split()
        chunks = []
        chunk_size = 160
        for index in range(0, len(words), chunk_size):
            chunks.append(clean_text(" ".join(words[index:index + chunk_size])))
        return [chunk for chunk in chunks if chunk]
    chunks = []
    current = ""
    for sentence in sentences:
        candidate = clean_text((current + " " + sentence).strip())
        if current and len(candidate) > target_chars:
            chunks.append(clean_text(current))
            current = sentence
        else:
            current = candidate
    if current:
        chunks.append(clean_text(current))
    normalized = []
    for chunk in chunks:
        if len(chunk) <= hard_limit:
            normalized.append(chunk)
            continue
        words = chunk.split()
        for index in range(0, len(words), 160):
            normalized.append(clean_text(" ".join(words[index:index + 160])))
    return [chunk for chunk in normalized if chunk]


def split_oversized_turns(turns: list[dict], *, target_chars: int = 950, hard_limit: int = 1800) -> list[dict]:
    normalized = []
    for turn in turns:
        text = clean_text(turn.get("text", ""))
        if not text:
            continue
        pieces = split_long_turn_text(text, target_chars=target_chars, hard_limit=hard_limit)
        for piece in pieces:
            normalized.append({**turn, "text": piece})
    return normalized


def trim_obvious_repetition(text: str) -> str:
    cleaned = clean_text(text)
    tokens = re.findall(r"[A-Za-z0-9']+", cleaned)
    if len(tokens) < 18:
        return cleaned
    for span in range(2, 8):
        phrase = tokens[-span:]
        if len(tokens) < span * 3:
            continue
        repeated = True
        for offset in range(2, 5):
            start = len(tokens) - span * offset
            end = len(tokens) - span * (offset - 1)
            if start < 0 or tokens[start:end] != phrase:
                repeated = False
                break
        if repeated:
            keep = tokens[: len(tokens) - span * 3] + phrase
            return clean_text(" ".join(keep))
    return cleaned


def is_obvious_hallucination(text: str) -> bool:
    tokens = re.findall(r"[A-Za-z0-9']+", clean_text(text).lower())
    if len(tokens) < 24:
        return False
    unique = set(tokens)
    if len(unique) <= 2:
        return True
    dominant_ratio = max(tokens.count(token) for token in unique) / len(tokens)
    return dominant_ratio >= 0.82 and len(unique) <= 5


def basic_turn_cleanup(turns: list[dict]) -> list[dict]:
    cleaned = []
    for turn in turns:
        text = trim_obvious_repetition(turn.get("text", ""))
        if not text or is_obvious_hallucination(text):
            continue
        speaker = clean_text(turn.get("speaker") or "Unknown") or "Unknown"
        if is_audience_like_turn(text):
            speaker = "Audience"
        cleaned.append(
            {
                "speaker": speaker,
                "start": float(turn.get("start", 0.0)),
                "end": float(turn.get("end", 0.0)),
                "text": text,
            }
        )
    merged = merge_adjacent_turns(cleaned, max_gap=1.25)
    return split_oversized_turns(merged, target_chars=750, hard_limit=1400)


def normalize_speaker_specs(known_speakers: list[Any]) -> list[dict]:
    specs = []
    for item in known_speakers or []:
        if isinstance(item, str):
            specs.append({"name": item, "aliases": [item]})
        elif isinstance(item, dict) and item.get("name"):
            aliases = stable_dedupe([item["name"], *list(item.get("aliases") or [])])
            specs.append({"name": item["name"], "aliases": aliases})
    return specs


def derive_allowed_speakers(metadata: dict) -> list[str]:
    names = []
    if metadata.get("primary_speaker"):
        names.append(metadata["primary_speaker"])
    for spec in normalize_speaker_specs(metadata.get("known_speakers") or []):
        names.append(spec["name"])
    names.extend(metadata.get("video_context", {}).get("likely_named_speakers") or [])
    if metadata.get("content_format") == "documentary":
        names.append("Narrator")
    if metadata.get("speaker_strategy") == "primary_plus_audience":
        names.append("Audience")
    return stable_dedupe(names)


def build_alias_map(metadata: dict) -> dict[str, str]:
    alias_map = {}
    for spec in normalize_speaker_specs(metadata.get("known_speakers") or []):
        canonical = spec["name"]
        for alias in spec["aliases"]:
            alias_map[re.sub(r"[^a-z0-9]+", "", alias.lower())] = canonical
    for name in derive_allowed_speakers(metadata):
        alias_map[re.sub(r"[^a-z0-9]+", "", name.lower())] = name
    return alias_map


def canonicalize_speaker_name(label: str, alias_map: dict[str, str]) -> str:
    cleaned = clean_text(label or "")
    if not cleaned:
        return "Unknown"
    key = re.sub(r"[^a-z0-9]+", "", cleaned.lower())
    return alias_map.get(key, cleaned)


def chunk_turns_for_refinement(turns: list[dict], target_chars: int = 7000) -> list[list[dict]]:
    normalized_turns = []
    per_turn_limit = max(900, int(target_chars * 0.45))
    for turn in turns:
        text = clean_text(turn.get("text", ""))
        if not text:
            continue
        pieces = split_long_turn_text(text, target_chars=min(950, per_turn_limit), hard_limit=per_turn_limit)
        for piece in pieces:
            normalized_turns.append({**turn, "text": piece})

    chunks = []
    current = []
    current_chars = 0
    for turn in normalized_turns:
        line = f"{turn['speaker']}: {turn['text']}"
        if current and current_chars + len(line) + 2 > target_chars:
            chunks.append(current)
            current = []
            current_chars = 0
        current.append(turn)
        current_chars += len(line) + 2
    if current:
        chunks.append(current)
    return chunks


def build_gemini_client():
    api_key = resolve_api_key()
    if not api_key:
        raise RuntimeError("Gemini refinement is enabled but no API key was found.")
    try:
        from google import genai
    except Exception as exc:
        raise RuntimeError(f"google.genai could not be imported: {exc}") from exc
    return genai.Client(api_key=api_key)


def build_refinement_prompt(metadata: dict, chunk_turns: list[dict], chunk_index: int, chunk_total: int) -> str:
    allowed_speakers = derive_allowed_speakers(metadata)
    rough_chunk = "\n\n".join(f"{turn['speaker']}: {turn['text']}" for turn in chunk_turns)
    return dedent(
        f'''
        Repair this rough transcript chunk.

        Hard requirements:
        - Keep the wording faithful to the source transcript chunk.
        - Do not summarize.
        - Do not sanitize or euphemize harmful, abusive, sexual, or violent wording if it is already in the transcript chunk.
        - Fix obvious speaker cutoffs where one sentence was split across the wrong speaker boundary.
        - Merge same-speaker fragments into natural readable turns.
        - Keep turns paragraph-sized and readable. Do not collapse an entire sermon, talk, or interview response into one massive turn.
        - Prefer multiple medium-length turns over one giant turn when the same speaker continues across multiple paragraphs or ideas.
        - Fix only high-confidence ASR mistakes. If unclear, keep the rough wording.
        - Do not invent new named speakers. Use only clearly supported names.
        - If a speaker is uncertain, keep a generic label like Speaker 1, Speaker 2, Narrator, Audience, or Unknown.
        - For sermon or talk content with one dominant voice, keep the dominant named speaker stable across consecutive turns unless there is a clearly different audience, worship, or inserted media voice.
        - Output JSON only with schema: {{"turns": [{{"speaker": "...", "text": "..."}}]}}

        Context:
        - title: {metadata.get("title")}
        - content format: {metadata.get("content_format")}
        - primary speaker: {metadata.get("primary_speaker")}
        - allowed preferred speaker names: {allowed_speakers}
        - local notes: {metadata.get("context_notes")}
        - video grounding notes: {metadata.get("video_context_notes")}
        - term hints: {metadata.get("term_hints")}
        - chunk: {chunk_index} of {chunk_total}

        Rough transcript chunk:
        {rough_chunk}
        '''
    ).strip()


def normalize_refined_turns(payload: Any) -> list[dict]:
    if isinstance(payload, dict):
        raw_turns = payload.get("turns") or []
    elif isinstance(payload, list):
        raw_turns = payload
    else:
        raw_turns = []
    normalized = []
    for item in raw_turns:
        if not isinstance(item, dict):
            continue
        speaker = clean_text(str(item.get("speaker") or "Unknown")) or "Unknown"
        text = clean_text(str(item.get("text") or ""))
        if not text:
            continue
        normalized.append({"speaker": speaker, "text": text})
    return normalized


def parse_transcript_style_lines(text: str) -> list[dict]:
    refined = []
    current = None
    for raw_line in (text or "").splitlines():
        line = raw_line.strip()
        if not line:
            continue
        match = re.match(r"^([^:\n]{1,80}):\s+(.*)$", line)
        if match:
            if current and clean_text(current.get("text", "")):
                refined.append({"speaker": clean_text(current["speaker"]) or "Unknown", "text": clean_text(current["text"])} )
            current = {"speaker": match.group(1), "text": match.group(2)}
            continue
        if current is not None:
            current["text"] = clean_text(current["text"] + " " + line)
    if current and clean_text(current.get("text", "")):
        refined.append({"speaker": clean_text(current["speaker"]) or "Unknown", "text": clean_text(current["text"])} )
    return refined


def fallback_refined_turns(text: str, chunk_turns: list[dict]) -> list[dict]:
    line_parsed = parse_transcript_style_lines(text)
    if line_parsed:
        return line_parsed
    return [{"speaker": turn.get("speaker") or "Unknown", "text": clean_text(turn.get("text", ""))} for turn in chunk_turns if clean_text(turn.get("text", ""))]


def refine_chunk_with_gemini(client, metadata: dict, chunk_turns: list[dict], chunk_index: int, chunk_total: int) -> list[dict]:
    from google.genai import types

    prompt = build_refinement_prompt(metadata, chunk_turns, chunk_index, chunk_total)
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            maxOutputTokens=GEMINI_MAX_OUTPUT_TOKENS,
            responseMimeType="application/json",
            safetySettings=build_gemini_safety_settings(),
        ),
    )
    text = extract_gemini_text(response)
    if not text:
        return fallback_refined_turns(text, chunk_turns)
    payload = parse_json_response(text, fallback={"turns": []})
    refined = normalize_refined_turns(payload)
    if not refined:
        refined = fallback_refined_turns(text, chunk_turns)
    if not refined:
        raise RuntimeError(f"Gemini returned no usable turns for transcript chunk {chunk_index}.")
    return refined


def apply_refined_turn_texts(refined_turns: list[dict]) -> list[dict]:
    normalized = []
    cursor = 0.0
    for item in refined_turns:
        text = clean_text(item.get("text", ""))
        if not text:
            continue
        normalized.append({"speaker": item.get("speaker") or "Unknown", "start": cursor, "end": cursor + 0.1, "text": text})
        cursor += 3.0
    return normalized


def validate_final_turns(turns: list[dict], metadata: dict) -> None:
    if not turns:
        raise RuntimeError("Transcript validation failed: no final turns were produced.")

    problematic = [turn for turn in turns if is_obvious_hallucination(turn.get("text", ""))]
    if problematic:
        raise RuntimeError("Transcript validation failed: hallucination-like turns remain after refinement.")

    content_format = (metadata.get("content_format") or "default").lower()
    generic_labels = {
        turn["speaker"]
        for turn in turns
        if turn["speaker"] == "Unknown" or str(turn["speaker"]).startswith("Speaker ")
    }

    if metadata.get("strict_known_speakers") and generic_labels:
        log.warning(
            "Strict known-speaker mode still has generic speakers %s. Keeping transcript instead of failing the job.",
            sorted(generic_labels),
        )

    if content_format in {"interview", "conversation_testimony"} and len(turns) < 6:
        raise RuntimeError("Transcript validation failed: too few turns for interview-style audio.")

    giant_turns = [turn for turn in turns if len(turn["text"]) > 3500]
    if giant_turns:
        raise RuntimeError("Transcript validation failed: at least one turn is still implausibly large.")


def finalize_speaker_labels(turns: list[dict], metadata: dict) -> list[dict]:
    alias_map = build_alias_map(metadata)
    finalized = []
    for turn in turns:
        speaker = canonicalize_speaker_name(turn.get("speaker") or "Unknown", alias_map)
        text = clean_text(turn.get("text", ""))
        if not text:
            continue
        if is_audience_like_turn(text):
            speaker = "Audience"
        finalized.append({**turn, "speaker": speaker, "text": text})
    merged = merge_adjacent_turns(finalized, max_gap=1.8, content_format=(metadata.get("content_format") or "default"))
    split_turns = split_oversized_turns(merged, target_chars=950, hard_limit=1800)
    validate_final_turns(split_turns, metadata)
    return split_turns


rough_turns = basic_turn_cleanup(turns)
print("Local turns before Gemini refinement:", len(rough_turns))

if USE_GEMINI_REFINEMENT:
    gemini_client = build_gemini_client()
    chunked_turns = chunk_turns_for_refinement(rough_turns, target_chars=GEMINI_CHUNK_TARGET_CHARS)
    refined_turns = []
    for chunk_index, chunk_turns in enumerate(chunked_turns, start=1):
        print(f"Refining transcript chunk {chunk_index}/{len(chunked_turns)} with Gemini...")
        refined_turns.extend(refine_chunk_with_gemini(gemini_client, metadata, chunk_turns, chunk_index, len(chunked_turns)))
    final_turns = finalize_speaker_labels(apply_refined_turn_texts(refined_turns), metadata)
else:
    final_turns = finalize_speaker_labels(rough_turns, metadata)

print("Final turns:", len(final_turns))
print("Speaker inventory:", sorted({turn["speaker"] for turn in final_turns}))
print("Turn preview:", final_turns[:5])


# Write transcript txt output


In [ ]:
def resolve_transcript_output_path(output_dir: Path, metadata: dict, transcript_filename: str | None) -> Path:
    output_dir = Path(output_dir).expanduser().resolve()
    if transcript_filename in (None, '', 'auto'):
        filename = safe_transcript_filename(metadata.get('title') or 'transcript')
    else:
        filename = transcript_filename if transcript_filename.lower().endswith('.txt') else f'{transcript_filename}.txt'
    return output_dir / filename


def write_transcript_txt(turns: list[dict], metadata: dict, output_path: str) -> str:
    output_path = str(Path(output_path).expanduser().resolve())
    title = metadata.get('title') or 'Untitled'
    turns = repair_sanitize_output_turns(turns)

    lines = [format_title_line(title)]

    stats_line = build_stats_line(metadata)
    if stats_line:
        lines.append(stats_line)

    speaker_name = metadata.get('primary_speaker')
    if speaker_name:
        lines.append(f'Speaker: {speaker_name}')

    lines.append('')

    for turn in turns:
        text = clean_text(turn.get('text', ''))
        if not text:
            continue
        speaker = turn.get('speaker') or 'Unknown'
        lines.append(f'{speaker}: {text}')
        lines.append('')

    transcript_text = '\n'.join(lines).rstrip() + '\n'
    Path(output_path).write_text(transcript_text, encoding='utf-8')
    return output_path


transcript_path = resolve_transcript_output_path(output_dir, metadata, TRANSCRIPT_FILENAME)
saved_path = write_transcript_txt(final_turns, metadata, str(transcript_path))

if not KEEP_TEMP:
    for artifact in artifacts_to_cleanup:
        try:
            artifact = Path(artifact)
            if artifact.exists() and artifact.is_file() and temp_dir in artifact.parents:
                artifact.unlink()
        except Exception as exc:
            log.warning('Could not remove temp artifact %s: %s', artifact, exc)

print('Saved transcript:', saved_path)


# Preview final transcript


In [ ]:
preview_text = Path(saved_path).read_text(encoding='utf-8')
print(preview_text[:4000])


# Troubleshooting notes

Common failure points on Mac:

- `ffmpeg` missing from PATH. Fix with `brew install ffmpeg`.
- `yt_dlp` missing in the notebook kernel. Fix with `pip install yt-dlp`.
- `pyannote.audio` import errors. Use Python 3.11, upgrade `pip`, and reinstall in the same environment as Jupyter.
- Hugging Face diarization access denied. Make sure:
  - `HF_TOKEN` is populated
  - you accepted terms for `pyannote/speaker-diarization-community-1`
- `google-genai` missing. Install from `requirements.txt` and set `GEMINI_API_KEY` or `GOOGLE_API_KEY` in `.env`.
- Gemini refinement is optional. If the key is missing, the notebook will still produce a local rough transcript.
- `mlx_whisper` install or API mismatch. Leave `ASR_BACKEND = "auto"` and let the notebook fall back to `faster-whisper`.
- `large-v3` feels too slow on a MacBook Air. Switch `WHISPER_MODEL` to `large-v3-turbo` for a practical speed-up.
- Documentary-style videos usually need wider diarization bounds than interviews. The notebook now derives those from `content_format`, but you can still override `NUM_SPEAKERS`, `MIN_SPEAKERS`, or `MAX_SPEAKERS` manually.
- If you want local audio plus real YouTube title and stats, set both `LOCAL_AUDIO_PATH` and `YOUTUBE_URL`. The notebook will use the local file and fetch metadata without redownloading audio.

Notes:

- The notebook saves one final `.txt` transcript per run. By default, the filename is derived from the video title.
- Intermediate Python structures for words, speaker segments, grounded context, and refined turns remain in memory only.
- The Gemini grounding stage uses the public YouTube URL directly, which helps with speaker naming and documentary/interview context.
- The Gemini refinement stage is intended for transcript restoration, not summarization. It is configured to preserve source wording rather than rewrite it.
